In [ ]:
!pip install geopandas

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import geopandas as gpd
file_path = '/content/drive/MyDrive/Online_Assesment_GISACT/Data/Building_Footprint.geojson'

# Membaca data
gdf_building = gpd.read_file(file_path)

#Mendeteksi anomali "bolong"
print("Total baris dan kolom:", gdf_building.shape)
print("\nJumlah missing values per kolom:")
print(gdf_building.isnull().sum())

Total baris dan kolom: (36642, 21)

Jumlah missing values per kolom:
id                        0
@id                       0
addr:city             35851
addr:full             35987
building                  0
building:condition    36034
building:floor        36048
building:levels       35595
building:roof         35859
building:structure    35860
building:walls        35861
capacity:persons      36014
name                  35591
name:vi               36634
office                36483
source                35973
type                  36630
access:roof           36075
amenity               36164
height                36579
geometry                  0
dtype: int64


In [ ]:
import pandas as pd
# Menyeleksi daftar kolom
kolom_ideal = [
    'id', '@id',
    'name',
    'building',
    'building:levels',
    'height',
    'building:roof',
    'amenity',
    'office',
    'capacity:persons',
    'geometry'
]

# Hanya mengambil kolom yang benar-benar ada
kolom_pilihan = [col for col in kolom_ideal if col in gdf_building.columns]

# Membuat GeoDataFrame baru yang bersih
gdf_clean = gdf_building[kolom_pilihan].copy()
print("Kolom yang dipertahankan:", gdf_clean.columns.tolist())

Kolom yang dipertahankan: ['id', '@id', 'name', 'building', 'building:levels', 'height', 'building:roof', 'amenity', 'office', 'capacity:persons', 'geometry']


In [ ]:
# Cek duplikat
print("Jumlah baris yang 100% duplikat:", gdf_clean.duplicated().sum())

# Cek duplikat berdasarkan ID unik
if '@id' in gdf_clean.columns:
    print("Jumlah duplikat pada ID Bangunan:", gdf_clean.duplicated(subset=['@id']).sum())

# Cek duplikat spasial
print("Jumlah duplikat pada Geometri:", gdf_clean.geometry.duplicated().sum())

Jumlah baris yang 100% duplikat: 0
Jumlah duplikat pada ID Bangunan: 0
Jumlah duplikat pada Geometri: 0


In [ ]:
# Mengganti atribute 'yes' menjadi 'building' di kolom 'building'
gdf_clean['building'] = gdf_clean['building'].replace('yes', 'building')

# Memeriksa daftar nilai unik (kategori)update
print("Kategori bangunan yang tersedia saat ini:")
print(gdf_clean['building'].unique())

Kategori bangunan yang tersedia saat ini:
['government_office' 'building' 'school' 'hospital' 'commercial' 'museum'
 'mosque' 'train_station' 'government' 'military' 'townhall' 'bank'
 'office' 'hotel' 'embassy' 'retail' 'marketplace' 'college' 'apartments'
 'offf' 'house' 'subdistrict_office' 'church' 'fire_station'
 'kindergarten' 'roof' 'garage' 'clinic' 'pumping_station' 'residential'
 'village_office' 'community_group_office' 'police' 'guest_house'
 'post_office' 'industrial' 'bridge' 'transportation' 'shed' 'pharmacy'
 'parking' 'governor_office' 'power_substation' 'public' 'restaurant'
 'temple' 'university' 'construction' 'detached' 'warehouse'
 'sports_centre' 'service']


In [ ]:
# Memperbaiki typo nilai 'offf' menjadi 'office' pada kolom 'building'
gdf_clean['building'] = gdf_clean['building'].replace('offf', 'office')

# DATA PROFILING: Memeriksa ulang untuk memastikan 'offf' sudah lenyap
print("Kategori bangunan setelah perbaikan typo:")
print(gdf_clean['building'].unique())

Kategori bangunan setelah perbaikan typo:
['government_office' 'building' 'school' 'hospital' 'commercial' 'museum'
 'mosque' 'train_station' 'government' 'military' 'townhall' 'bank'
 'office' 'hotel' 'embassy' 'retail' 'marketplace' 'college' 'apartments'
 'house' 'subdistrict_office' 'church' 'fire_station' 'kindergarten'
 'roof' 'garage' 'clinic' 'pumping_station' 'residential' 'village_office'
 'community_group_office' 'police' 'guest_house' 'post_office'
 'industrial' 'bridge' 'transportation' 'shed' 'pharmacy' 'parking'
 'governor_office' 'power_substation' 'public' 'restaurant' 'temple'
 'university' 'construction' 'detached' 'warehouse' 'sports_centre'
 'service']


In [ ]:
# MENGISI DATA YANG BOLONG (IMPUTASI)

# Mengisi data jumlah lantai dan tinggi
if 'building:levels' in gdf_clean.columns:
    gdf_clean['building:levels'] = gdf_clean['building:levels'].fillna(1).astype(int)

# Nilai high diasumsikan dengan megacu pada building levels, 1 lantai = 3 meter
    gdf_clean['height'] = gdf_clean['height'].fillna(gdf_clean['building:levels'] * 3.0)

# PERBAIKAN GEOMETRI (TOPOLOGY CLEANSING)
# Memastikan tidak ada poligon cacat yang akan mengganggu hitungan luas atap
gdf_clean['geometry'] = gdf_clean['geometry'].make_valid()


# DATA PROFILING AKHIR
print("\n2. Total baris dan kolom sekarang:", gdf_clean.shape)
print("\n3. Sisa data kosong:")
print(gdf_clean.isnull().sum())


2. Total baris dan kolom sekarang: (36642, 11)

3. Sisa data kosong:
id                      0
@id                     0
name                35591
building                0
building:levels         0
height                  0
building:roof       35859
amenity             36164
office              36483
capacity:persons    36014
geometry                0
dtype: int64


In [ ]:
# Mengecek struktur dan tipe data (Data Types) dari setiap kolom
print("--- INFORMASI STRUKTUR & TIPE DATA ---")
gdf_clean.info()

--- INFORMASI STRUKTUR & TIPE DATA ---
<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 36642 entries, 0 to 36641
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype   
---  ------            --------------  -----   
 0   id                36642 non-null  object  
 1   @id               36642 non-null  object  
 2   name              1051 non-null   object  
 3   building          36642 non-null  object  
 4   building:levels   36642 non-null  int64   
 5   height            36642 non-null  object  
 6   building:roof     783 non-null    object  
 7   amenity           478 non-null    object  
 8   office            159 non-null    object  
 9   capacity:persons  628 non-null    object  
 10  geometry          36642 non-null  geometry
dtypes: geometry(1), int64(1), object(9)
memory usage: 3.1+ MB


In [ ]:
# Mengonversi kolom height menjadi angka (jika kolom tersebut sudah ada dan terbaca teks)
if 'height' in gdf_clean.columns:
    gdf_clean['height'] = pd.to_numeric(gdf_clean['height'], errors='coerce')
    print("Berhasil: 'height' telah diubah ke format angka.")

# Menampilkan tipe data khusus untuk kolom yang baru saja diubah
kolom_cek = [col for col in ['height'] if col in gdf_clean.columns]
print(gdf_clean[kolom_cek].dtypes)


Berhasil: 'height' telah diubah ke format angka.
height    float64
dtype: object


In [ ]:
gdf_clean.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 36642 entries, 0 to 36641
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype   
---  ------            --------------  -----   
 0   id                36642 non-null  object  
 1   @id               36642 non-null  object  
 2   name              1051 non-null   object  
 3   building          36642 non-null  object  
 4   building:levels   36642 non-null  int64   
 5   height            36642 non-null  float64 
 6   building:roof     783 non-null    object  
 7   amenity           478 non-null    object  
 8   office            159 non-null    object  
 9   capacity:persons  628 non-null    object  
 10  geometry          36642 non-null  geometry
dtypes: float64(1), geometry(1), int64(1), object(8)
memory usage: 3.1+ MB


In [ ]:
# Menampilkan KESELURUHAN tabel atribut
from google.colab import data_table
data_table.enable_dataframe_formatter()

print("\n--- KESELURUHAN TABEL ATRIBUT ---")
gdf_clean


--- KESELURUHAN TABEL ATRIBUT ---


,id,@id,name,building,building:levels,height,building:roof,amenity,office,capacity:persons,geometry
0,relation/7237601,relation/7237601,Kementerian Pertahanan Republik Indonesia,government_office,2,6.0,concrete,None,government,>500,"POLYGON ((106.82228 -6.17692, 106.82232 -6.176..."
1,relation/7247047,relation/7247047,None,building,1,3.0,None,None,None,None,"POLYGON ((106.82629 -6.16914, 106.82627 -6.168..."
2,relation/7268563,relation/7268563,SD Negeri Menteng 01 Jakarta,school,1,3.0,tile,school,None,50-100,"POLYGON ((106.82889 -6.19851, 106.82884 -6.198..."
3,relation/7296736,relation/7296736,None,school,2,6.0,asbestos,None,None,250-500,"MULTIPOLYGON (((106.81782 -6.18416, 106.81796 ..."
4,relation/7333422,relation/7333422,Rawat Inap RSAB Harapan Kita,hospital,4,12.0,concrete,None,None,>500,"POLYGON ((106.79835 -6.18507, 106.79835 -6.185..."
...,...,...,...,...,...,...,...,...,...,...,...
36637,way/1514023978,way/1514023978,None,office,15,45.0,None,None,None,None,"POLYGON ((106.82368 -6.18746, 106.82368 -6.188..."
36638,way/1514023981,way/1514023981,Polsubsektor Sarinah,building,2,6.0,None,police,None,None,"POLYGON ((106.82422 -6.18835, 106.82422 -6.188..."
36639,way/1514023984,way/1514023984,None,roof,1,3.0,None,None,None,None,"POLYGON ((106.82442 -6.18836, 106.82442 -6.188..."
36640,way/655649064,way/655649064,Menara Radius Prawiro,commercial,25,75.0,None,None,None,None,"POLYGON ((106.82134 -6.18234, 106.8218 -6.1823..."


In [ ]:
# Tentukan lokasi penyimpanan file hasil pembersihan
output_path = '/content/drive/MyDrive/Online_Assesment_GISACT/Data/Building_Footprint_Cleaned_fix.geojson'

# Proses ekspor ke format GeoJSON
gdf_clean.to_file(output_path, driver='GeoJSON')

print(f"Data bangunan yang sudah bersih berhasil disimpan di: {output_path}")

Data bangunan yang sudah bersih berhasil disimpan di: /content/drive/MyDrive/Online_Assesment_GISACT/Data/Building_Footprint_Cleaned_fix.geojson


In [ ]:
# Load data
gdf_road = gpd.read_file('/content/drive/MyDrive/Online_Assesment_GISACT/Data/main_road.geojson')

# Lihat dimensi dan daftar kolom
print("Total baris dan kolom jalan:", gdf_road.shape)
print("\nDaftar atribut yang tersedia:")
print(gdf_road.columns.tolist())

# Cek contoh datanya agar kita tahu isi kolomnya
print("\n--- CONTOH DATA JALAN ---")
display(gdf_road.head())

Total baris dan kolom jalan: (780, 67)

Daftar atribut yang tersedia:
['id', '@id', 'access:conditional', 'alt_name', 'bicycle', 'bridge', 'bus', 'bus:lanes', 'busway:left', 'busway:right', 'covered', 'cycleway:both', 'cycleway:left', 'cycleway:right', 'description', 'destination', 'destination:lanes', 'destination:ref', 'est_width', 'foot', 'hgv', 'highway', 'horse', 'int_ref', 'junction', 'lane_markings', 'lanes', 'lanes:backward', 'lanes:forward', 'layer', 'lit', 'maxheight', 'maxspeed', 'maxspeed:type', 'maxweight', 'maxweight:signed', 'minspeed', 'motor_vehicle', 'motorcar', 'motorcycle', 'name', 'name:etymology:wikidata', 'name:etymology:wikipedia', 'name:id', 'note', 'oneway', 'oneway:conditional', 'priority_road', 'psv:conditional', 'ref', 'sidewalk', 'sidewalk:both', 'sidewalk:both:bicycle', 'sidewalk:left', 'sidewalk:left:bicycle', 'sidewalk:right', 'smoothness', 'source', 'source:maxspeed', 'surface', 'toll', 'tunnel', 'turn:lanes', 'vehicle:conditional', 'wheelchair', 'widt

,id,@id,access:conditional,alt_name,bicycle,bridge,bus,bus:lanes,busway:left,busway:right,...,source,source:maxspeed,surface,toll,tunnel,turn:lanes,vehicle:conditional,wheelchair,width,geometry
0,way/11591853,way/11591853,None,None,use_sidepath,None,None,None,None,None,...,None,None,asphalt,None,None,None,None,None,None,"LINESTRING (106.82309 -6.20173, 106.82299 -6.2..."
1,way/11668561,way/11668561,None,None,None,yes,None,None,None,None,...,None,None,asphalt,None,None,None,None,None,12.57,"LINESTRING (106.82802 -6.20267, 106.82801 -6.2..."
2,way/11669819,way/11669819,None,None,None,None,None,None,None,None,...,HOT_InAWARESurvey_2017,None,asphalt,None,None,None,None,None,8,"LINESTRING (106.81999 -6.17655, 106.81993 -6.1..."
3,way/11670523,way/11670523,None,None,None,None,None,None,None,None,...,None,None,asphalt,None,None,None,None,None,None,"LINESTRING (106.83186 -6.1792, 106.83184 -6.17..."
4,way/18056345,way/18056345,None,None,None,None,None,None,None,None,...,HOT_InAWARESurvey_2017,None,asphalt,None,None,None,None,None,9,"LINESTRING (106.82575 -6.18315, 106.82592 -6.1..."


In [ ]:
# Daftar kolom (Whitelist)
kolom_jalan_ideal = ['id', '@id', 'name', 'highway', 'oneway', 'lanes', 'surface', 'width', 'geometry']

# Ambil hanya kolom yang ada di dataset
kolom_jalan_pilihan = [col for col in kolom_jalan_ideal if col in gdf_road.columns]
gdf_road_clean = gdf_road[kolom_jalan_pilihan].copy()

# Profiling Missing Values
print("--- HASIL WHITELIST ---")
print("Kolom yang tersisa:", gdf_road_clean.columns.tolist())
print("\n--- CEK DATA BOLONG (NULL) ---")
print(gdf_road_clean.isnull().sum())

# Melihat Variasi Klasifikasi Jalan
print("\n--- KLASIFIKASI JALAN YANG TERSEDIA ---")
print(gdf_road_clean['highway'].value_counts())

--- HASIL WHITELIST ---
Kolom yang tersisa: ['id', '@id', 'name', 'highway', 'oneway', 'lanes', 'surface', 'width', 'geometry']

--- CEK DATA BOLONG (NULL) ---
id            0
@id           0
name         18
highway       0
oneway        1
lanes         0
surface       0
width       284
geometry      0
dtype: int64

--- KLASIFIKASI JALAN YANG TERSEDIA ---
highway
primary      590
secondary    108
trunk         53
motorway      29
Name: count, dtype: int64


In [ ]:
# Cek baris duplikat total
print(f"Jumlah baris duplikat: {gdf_road_clean.duplicated().sum()}")

# Cek apakah ada ID jalan yang kembar (jika '@id' tersedia)
if '@id' in gdf_road_clean.columns:
    print(f"Jumlah ID jalan duplikat: {gdf_road_clean.duplicated(subset=['@id']).sum()}")

# Jika ditemukan duplikat, kita hapus
gdf_road_clean = gdf_road_clean.drop_duplicates()

Jumlah baris duplikat: 0
Jumlah ID jalan duplikat: 0


In [ ]:
# Mengecek tipe data setiap kolom
print("--- TIPE DATA JALAN ---")
print(gdf_road_clean.info())

--- TIPE DATA JALAN ---
<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 780 entries, 0 to 779
Data columns (total 9 columns):
 #   Column    Non-Null Count  Dtype   
---  ------    --------------  -----   
 0   id        780 non-null    object  
 1   @id       780 non-null    object  
 2   name      762 non-null    object  
 3   highway   780 non-null    object  
 4   oneway    779 non-null    object  
 5   lanes     780 non-null    object  
 6   surface   780 non-null    object  
 7   width     496 non-null    object  
 8   geometry  780 non-null    geometry
dtypes: geometry(1), object(8)
memory usage: 55.0+ KB
None


In [ ]:
gdf_road_clean['width'] = pd.to_numeric(gdf_road_clean['width'], errors='coerce')

print("\n--- TIPE DATA SETELAH DIUBAH ---")
print(gdf_road_clean[['width']].info())


--- TIPE DATA SETELAH DIUBAH ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 780 entries, 0 to 779
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   width   496 non-null    float64
dtypes: float64(1)
memory usage: 6.2 KB
None


In [ ]:
print(gdf_road_clean.info())

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 780 entries, 0 to 779
Data columns (total 9 columns):
 #   Column    Non-Null Count  Dtype   
---  ------    --------------  -----   
 0   id        780 non-null    object  
 1   @id       780 non-null    object  
 2   name      762 non-null    object  
 3   highway   780 non-null    object  
 4   oneway    779 non-null    object  
 5   lanes     780 non-null    object  
 6   surface   780 non-null    object  
 7   width     496 non-null    float64 
 8   geometry  780 non-null    geometry
dtypes: float64(1), geometry(1), object(7)
memory usage: 55.0+ KB
None


In [ ]:
if 'width' in gdf_clean.columns:
    gdf_clean['width'] = pd.to_numeric(gdf_clean['width'], errors='coerce')
    print("Berhasil: 'width' telah diubah ke format angka.")

In [ ]:
gdf_road_clean['lanes'] = pd.to_numeric(gdf_road_clean['lanes'], errors='coerce')
gdf_road_clean['width'] = pd.to_numeric(gdf_road_clean['width'], errors='coerce')

# Isi lanes kosong dengan 1
gdf_road_clean['lanes'] = gdf_road_clean['lanes'].fillna(1).astype(int)

# Isi surface kosong dengan 'asphalt'
gdf_road_clean['surface'] = gdf_road_clean['surface'].fillna('asphalt')

# 3. Cek hasil akhir
print("--- STATUS DATA JALAN ---")
print(gdf_road_clean.isnull().sum())

--- STATUS DATA JALAN ---
id            0
@id           0
name         18
highway       0
oneway        1
lanes         0
surface       0
width       284
geometry      0
dtype: int64


In [ ]:
# Tentukan lokasi penyimpanan file jalan hasil pembersihan
output_path_road = '/content/drive/MyDrive/Online_Assesment_GISACT/Data/Main_Road_Cleaned_fix.geojson'

# Proses ekspor ke format GeoJSON
gdf_road_clean.to_file(output_path_road, driver='GeoJSON')

print(f"Data jalan yang sudah bersih dan terklasifikasi berhasil disimpan di: {output_path_road}")

Data jalan yang sudah bersih dan terklasifikasi berhasil disimpan di: /content/drive/MyDrive/Online_Assesment_GISACT/Data/Main_Road_Cleaned_fix.geojson


In [ ]:
# Load data POI
gdf_poi = gpd.read_file('/content/drive/MyDrive/Online_Assesment_GISACT/Data/POI.geojson')

# 1. Cek dimensi dan kolom
print("Total baris dan kolom POI:", gdf_poi.shape)
print("\nDaftar kolom POI:")
print(gdf_poi.columns.tolist())

# 2. Lihat contoh isi datanya
display(gdf_poi.head())

Total baris dan kolom POI: (433, 114)

Daftar kolom POI:
['id', '@id', 'addr:city', 'addr:district', 'addr:floor', 'addr:full', 'addr:housename', 'addr:housenumber', 'addr:neighbourhood', 'addr:postcode', 'addr:province', 'addr:street', 'addr:subdistrict', 'air_conditioning', 'alt_name', 'amenity', 'atm', 'branch', 'brand', 'brand:en', 'brand:id', 'brand:website', 'brand:wikidata', 'brand:wikipedia', 'brand:zh', 'capacity', 'check_date', 'check_date:opening_hours', 'contact:email', 'contact:facebook', 'contact:instagram', 'created_by', 'cuisine', 'delivery', 'delivery:gofood', 'delivery:gofood:ref', 'delivery:grabfood', 'delivery:grabfood:ref', 'diet:halal', 'diet:healthy', 'diet:meat', 'diet:vegan', 'diet:vegetarian', 'drive_through', 'email', 'healthcare', 'highchair', 'indoor_seating', 'internet_access', 'internet_access:fee', 'internet_access:ssid', 'level', 'name', 'name:en', 'name:fr', 'name:id', 'name:ja', 'name:nl', 'name:zh', 'notes', 'office', 'official_name', 'official_name:

,id,@id,addr:city,addr:district,addr:floor,addr:full,addr:housename,addr:housenumber,addr:neighbourhood,addr:postcode,...,toilets,toilets:access,toilets:wheelchair,tourism,website,wheelchair,wikidata,wikipedia,wikipedia:en,geometry
0,node/309939284,node/309939284,None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,POINT (106.81561 -6.19469)
1,node/824317972,node/824317972,None,None,None,None,None,None,None,None,...,None,None,None,hotel,None,None,None,None,None,POINT (106.82918 -6.18576)
2,node/824318011,node/824318011,None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,POINT (106.82927 -6.18531)
3,node/1091887771,node/1091887771,None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,POINT (106.82362 -6.18804)
4,node/1354554844,node/1354554844,None,None,None,None,None,None,None,None,...,None,None,None,None,None,yes,None,None,None,POINT (106.79638 -6.18923)


In [ ]:
# 1. Daftar kolom (Whitelist)
kolom_poi_ideal = ['id', 'name', 'amenity', 'shop', 'office', 'tourism', 'healthcare', 'geometry']

# 2. Ambil hanya kolom yang ada di dataset
kolom_poi_pilihan = [col for col in kolom_poi_ideal if col in gdf_poi.columns]
gdf_poi_clean = gdf_poi[kolom_poi_pilihan].copy()

# 3. Normalisasi: Gabungkan fungsi menjadi satu kolom 'poi_type'
def tentukan_tipe_poi(row):
    if pd.notnull(row['amenity']): return row['amenity']
    if pd.notnull(row['shop']): return row['shop']
    if pd.notnull(row['office']): return row['office']
    if pd.notnull(row['tourism']): return row['tourism']
    if pd.notnull(row['healthcare']): return row['healthcare']
    return 'unknown'

gdf_poi_clean['poi_type'] = gdf_poi_clean.apply(tentukan_tipe_poi, axis=1)

# 4. Hapus kolom sumber lama karena sudah digabung
gdf_poi_clean = gdf_poi_clean.drop(columns=['amenity', 'shop', 'office', 'tourism', 'healthcare'])

# 5. Cek hasil
print("--- HASIL PEMBERSIHAN POI ---")
print(gdf_poi_clean['poi_type'].value_counts().head(10))

--- HASIL PEMBERSIHAN POI ---
poi_type
bank           153
restaurant     134
cafe            69
hotel           62
hospital         8
supermarket      7
Name: count, dtype: int64


In [ ]:
# Mengecek tipe data setiap kolom
print("--- TIPE DATA POI ---")
print(gdf_poi_clean.info())

--- TIPE DATA POI ---
<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 433 entries, 0 to 432
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype   
---  ------    --------------  -----   
 0   id        433 non-null    object  
 1   name      429 non-null    object  
 2   geometry  433 non-null    geometry
 3   poi_type  433 non-null    object  
dtypes: geometry(1), object(3)
memory usage: 13.7+ KB
None


In [ ]:
# Tentukan lokasi penyimpanan file POI hasil pembersihan
output_path_poi = '/content/drive/MyDrive/Online_Assesment_GISACT/Data/POI_Cleaned_fix.geojson'

# Proses ekspor ke format GeoJSON
gdf_poi_clean.to_file(output_path_poi, driver='GeoJSON')

print(f"Data poi yang sudah bersih dan terklasifikasi berhasil disimpan di: {output_path_poi}")

Data poi yang sudah bersih dan terklasifikasi berhasil disimpan di: /content/drive/MyDrive/Online_Assesment_GISACT/Data/POI_Cleaned_fix.geojson


In [ ]:
import geopandas as gpd

gdf_building_clean = gpd.read_file('/content/drive/MyDrive/Online_Assesment_GISACT/Data/Building_Footprint_Cleaned_fix.geojson')
gdf_road_clean = gpd.read_file('/content/drive/MyDrive/Online_Assesment_GISACT/Data/Main_Road_Cleaned_fix.geojson')
gdf_poi_clean = gpd.read_file('/content/drive/MyDrive/Online_Assesment_GISACT/Data/POI_Cleaned_fix.geojson')

print("Data cleaned berhasil di upload")

Data cleaned berhasil di upload


In [ ]:
print("--- CEK CRS AWAL ---")
print("CRS Bangunan:", gdf_building_clean.crs)
print("CRS Jalan:", gdf_road_clean.crs)
print("CRS POI:", gdf_poi_clean.crs)

--- CEK CRS AWAL ---
CRS Bangunan: EPSG:4326
CRS Jalan: EPSG:4326
CRS POI: EPSG:4326
